# Cell 1. Imports

In [0]:
import gzip
import io
import json
import uuid
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List

import boto3
import requests

# Cell 2. Widgets (Parameters)

In [0]:
dbutils.widgets.text("series_ids", "DFF,CPIAUCSL,DTWEXBGS")  # List of Series IDs
dbutils.widgets.text("api_key", "", "FRED API Key")
dbutils.widgets.dropdown("mode", "realtime", ["hist_full", "realtime"])

# ---- Date window (optional overrides, mainly for hist_full chunking) ----
dbutils.widgets.text("start_date", "")  # YYYY-MM-DD
dbutils.widgets.text("end_date", "")  # YYYY-MM-DD

# ---- S3 target ----
dbutils.widgets.text("s3_bucket", "enterprise-raw-lakehouse")
dbutils.widgets.text("s3_base_prefix", "macro/fred")  # Base path

# ---- AWS Credentials ----
dbutils.widgets.text("aws_access_key", "", "AWS Access Key")
dbutils.widgets.text("aws_secret_key", "", "AWS Secret Key")
dbutils.widgets.text("aws_region", "eu-central-1", "AWS Region")

print("✅ Widgets created. Please configure API Key and AWS credentials.")

# Cell 3. Configuration & Setup

In [0]:
# 1. Parse Parameters
SERIES_IDS = [s.strip().upper() for s in dbutils.widgets.get("series_ids").split(",") if s.strip()]
API_KEY = dbutils.widgets.get("api_key").strip()
MODE = dbutils.widgets.get("mode").strip().lower()

START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE = dbutils.widgets.get("end_date").strip()

BUCKET_NAME = dbutils.widgets.get("s3_bucket").strip()
S3_BASE_PREFIX = dbutils.widgets.get("s3_base_prefix").strip().strip("/")
ACCESS_KEY = dbutils.widgets.get("aws_access_key").strip()
SECRET_KEY = dbutils.widgets.get("aws_secret_key").strip()
REGION = dbutils.widgets.get("aws_region").strip()

if not SERIES_IDS:
    raise ValueError("❌ series_ids is empty.")
if not API_KEY:
    raise ValueError("❌ FRED api_key is required.")
if not ACCESS_KEY or not SECRET_KEY:
    raise ValueError("❌ Missing AWS Keys.")

# 2. Path Routing Map (Define how Series IDs map to S3 folders)
PATH_MAP = {
    "DFF": "interest_rate/fedfunds",
    "FEDFUNDS": "interest_rate/fedfunds",
    "CPIAUCSL": "inflation/cpi",
    "DTWEXBGS": "fx/dxy",
    # Default fallback if not found below
}

# 3. Mode Logic (Determine effective dates if none provided)
# FRED updates relatively slowly. 'realtime' can just be the last 30 days to ensure recent revisions are caught.
effective_start = START_DATE
effective_end = END_DATE

if MODE == "realtime" and not START_DATE:
    # Fetch last 30 days for safety to catch revisions
    effective_start = (datetime.now(timezone.utc) - timedelta(days=30)).strftime("%Y-%m-%d")

print(f"🚀 PRODUCER CONFIG | MODE: {MODE}")
print(f"🔹 Series: {SERIES_IDS}")
print(f"🔹 S3 Base: s3://{BUCKET_NAME}/{S3_BASE_PREFIX}/")
print(f"🔹 Window: Start={effective_start or 'Origin'}, End={effective_end or 'Now'}")

# Cell 4. Initialize S3 Client


In [0]:
_boto_sess = boto3.session.Session(
    aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY, region_name=REGION
)
s3 = _boto_sess.client("s3")
print("✅ S3 client ready.")

# Cell 5.  Extraction & Upload Logic

In [0]:
def _to_iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def upload_jsonl_gz(bucket: str, key: str, rows: List[Dict[str, Any]]):
    if not rows:
        return
    buf = io.BytesIO()
    with gzip.GzipFile(fileobj=buf, mode="wb") as gz:
        for r in rows:
            line = json.dumps(r, ensure_ascii=False) + "\n"
            gz.write(line.encode("utf-8"))

    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=buf.getvalue(),
        ContentType="application/x-ndjson",
        ContentEncoding="gzip",
    )
    print(f"   💾 Uploaded to s3://{bucket}/{key}")


def build_s3_key(series_id: str, ingest_dt: datetime) -> str:
    dt_str = ingest_dt.strftime("%Y-%m-%d")
    hhmmss = ingest_dt.strftime("%H%M%S")

    # Resolve category based on series ID, default to "other"
    category_path = PATH_MAP.get(series_id, f"other/{series_id.lower()}")

    # Result: macro/fred/interest_rate/fedfunds/dt=YYYY-MM-DD/DFF_hist_full_YYYYMMDDT...Z.jsonl.gz
    return f"{S3_BASE_PREFIX}/{category_path}/dt={dt_str}/{series_id}_{MODE}_{dt_str}T{hhmmss}Z.jsonl.gz"

# Cell 6. Execute Production

In [0]:
BASE_URL = "https://api.stlouisfed.org/fred/series/observations"
ingestion_ts = datetime.now(timezone.utc)
run_id = str(uuid.uuid4())

print(f"\n--- ⏳ Starting Extraction (Run ID: {run_id}) ---")

for sid in SERIES_IDS:
    print(f"\n📥 Processing {sid}...")
    try:
        params = {"series_id": sid, "api_key": API_KEY, "file_type": "json"}
        if effective_start:
            params["observation_start"] = effective_start
        if effective_end:
            params["observation_end"] = effective_end

        r = requests.get(BASE_URL, params=params, timeout=30)
        r.raise_for_status()

        # Raw dataset payload
        raw_payload = json.dumps(r.json(), separators=(",", ":"))

        record = {
            "series_id": sid,
            "raw_json": raw_payload,
            "run_id": run_id,
            "ingestion_ts": _to_iso_z(ingestion_ts),
            "mode": MODE,
            "source": "fred",
        }

        key = build_s3_key(sid, ingestion_ts)
        upload_jsonl_gz(BUCKET_NAME, key, [record])
        print(f"   ✅ [OK] {sid}")

    except Exception as e:
        print(f"   ❌ [ERROR] Failed to fetch {sid}: {e}")

print("\n🎉 Producer pipeline finished.")